In [ ]:
import json
import re
from collections import Counter
import numpy as np
import pandas as pd


def normalize_hebrew(text):
    if pd.isna(text) or text is None:
        return ""
    text = re.sub(r"[^\w\s]", "", str(text))
    return " ".join(text.strip().split())


def normalize_class_val(val):
    """Treats 'null', 'None', 'NaN', None, and empty strings as equivalent."""
    if pd.isna(val) or val is None:
        return ""
    val_str = str(val).strip().lower()
    if val_str in ["null", "none", "nan", ""]:
        return ""
    return str(val).strip()


def token_f1(pred, gold):
    p_tokens = normalize_hebrew(pred).split()
    g_tokens = normalize_hebrew(gold).split()
    if not p_tokens and not g_tokens:
        return 1.0
    if not p_tokens or not g_tokens:
        return 0.0
    common = sum((Counter(p_tokens) & Counter(g_tokens)).values())
    if common == 0:
        return 0.0
    prec = common / len(p_tokens)
    rec = common / len(g_tokens)
    return 2 * prec * rec / (prec + rec)


# ==============================================================================
# 1. LOAD DATASETS
# ==============================================================================

json_files = ["results1.json", "results2.json", "results3.json", "results4.json"]
combined_json_data = []

for file_name in json_files:
    try:
        with open(file_name, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                combined_json_data.extend(data)
            else:
                combined_json_data.append(data)
    except FileNotFoundError:
        pass

gold_df = pd.read_excel("test_set_answers.xlsx")
gold_df["Original_ID"] = gold_df["Original_ID"].astype(str).str.strip()

STAGE3_HEADS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

# Flatten predictions with standardized null normalization
pred_rows = []
for item in combined_json_data:
    orig_id = str(item.get("Original_ID")).strip()
    sentence = item.get("sentence", "")
    for res in item.get("results", []):
        cls = res.get("classification", {})
        row = {
            "Original_ID": orig_id,
            "sentence": sentence,
            "meaning_value": str(res.get("meaning_value", "")).strip(),
            "referent": str(res.get("referent", "")).strip(),
        }
        for head in STAGE3_HEADS:
            row[head] = normalize_class_val(cls.get(head))
        pred_rows.append(row)

pred_df = pd.DataFrame(pred_rows)

# ==============================================================================
# 2. STAGE 1 EVALUATION
# ==============================================================================

s1_tp, s1_fp, s1_fn = 0, 0, 0
exact_sentence_matches = 0
total_sentences = len(combined_json_data)

for item in combined_json_data:
    orig_id = str(item.get("Original_ID")).strip()
    gold_mvs = set(
        gold_df[gold_df["Original_ID"] == orig_id]["MeaningVal"]
        .dropna()
        .apply(normalize_hebrew)
    )
    pred_mvs = set(
        [
            normalize_hebrew(r.get("meaning_value"))
            for r in item.get("results", [])
        ]
    )

    s1_tp += len(gold_mvs & pred_mvs)
    s1_fp += len(pred_mvs - gold_mvs)
    s1_fn += len(gold_mvs - pred_mvs)
    if gold_mvs and gold_mvs == pred_mvs:
        exact_sentence_matches += 1

s1_prec = s1_tp / max(s1_tp + s1_fp, 1)
s1_rec = s1_tp / max(s1_tp + s1_fn, 1)
s1_f1 = (
    2 * s1_prec * s1_rec / max(s1_prec + s1_rec, 1e-9)
    if (s1_prec + s1_rec) > 0
    else 0
)

# ==============================================================================
# 3. STAGE 2 & STAGE 3 (ORACLE VS. END-TO-END)
# ==============================================================================

total_gold_rows = len(gold_df.dropna(subset=["MeaningVal"]))

oracle_s2_exact, oracle_s2_total = 0, 0
oracle_s2_f1s = []
oracle_s3_correct = {head: 0 for head in STAGE3_HEADS}
oracle_s3_totals = {head: 0 for head in STAGE3_HEADS}
oracle_joint_correct, oracle_joint_total = 0, 0

e2e_s2_exact = 0
e2e_s2_f1s = []
e2e_s3_correct = {head: 0 for head in STAGE3_HEADS}
e2e_joint_correct = 0

for _, pred_row in pred_df.iterrows():
    orig_id = str(pred_row["Original_ID"]).strip()
    norm_mv = normalize_hebrew(pred_row["meaning_value"])

    gold_subset = gold_df[gold_df["Original_ID"] == orig_id]
    gold_match = gold_subset[
        gold_subset["MeaningVal"].apply(normalize_hebrew) == norm_mv
    ]

    if gold_match.empty:
        continue

    g_row = gold_match.iloc[0]

    # Stage 2
    p_ref = normalize_hebrew(pred_row["referent"])
    g_ref = normalize_hebrew(g_row["Referent"])
    tf1 = token_f1(p_ref, g_ref)

    oracle_s2_f1s.append(tf1)
    e2e_s2_f1s.append(tf1)
    oracle_s2_total += 1

    if p_ref == g_ref:
        oracle_s2_exact += 1
        e2e_s2_exact += 1

    # Stage 3
    all_7_correct = True
    for head in STAGE3_HEADS:
        p_val = normalize_class_val(pred_row[head])
        g_val = normalize_class_val(g_row[head])

        oracle_s3_totals[head] += 1
        if p_val == g_val:
            oracle_s3_correct[head] += 1
            e2e_s3_correct[head] += 1
        else:
            all_7_correct = False

    oracle_joint_total += 1
    if all_7_correct:
        oracle_joint_correct += 1
        e2e_joint_correct += 1

# ==============================================================================
# 4. PRINT SUMMARY
# ==============================================================================

print("\n" + "=" * 65)
print("             ORACLE VS. END-TO-END (UNIFIED NULLS)               ")
print("=" * 65)

print("\n--- STAGE 3: ASPECT CLASSIFICATION ACCURACY ---")
print(f"{'Aspect':12s} | {'Oracle Accuracy':20s} | {'E2E Accuracy':20s}")
print("-" * 65)

oracle_accs, e2e_accs = [], []
for head in STAGE3_HEADS:
    o_acc = (
        (oracle_s3_correct[head] / oracle_s3_totals[head]) * 100
        if oracle_s3_totals[head] > 0
        else 0.0
    )
    e_acc = (
        (e2e_s3_correct[head] / total_gold_rows) * 100
        if total_gold_rows > 0
        else 0.0
    )

    oracle_accs.append(o_acc)
    e2e_accs.append(e_acc)

    o_str = f"{o_acc:.2f}% ({oracle_s3_correct[head]}/{oracle_s3_totals[head]})"
    e_str = f"{e_acc:.2f}% ({e2e_s3_correct[head]}/{total_gold_rows})"
    print(f"{head:12s} | {o_str:20s} | {e_str:20s}")

print("-" * 65)
print(
    f"Macro-Average  | {np.mean(oracle_accs):.2f}%"
    f"{'':13s} | {np.mean(e2e_accs):.2f}%"
)
print(
    f"Strict Joint   | "
    f"{(oracle_joint_correct/max(oracle_joint_total, 1))*100:.2f}%"
    f"{'':13s} | {(e2e_joint_correct/max(total_gold_rows, 1))*100:.2f}%"
)
print("=" * 65)


             ORACLE VS. END-TO-END (UNIFIED NULLS)               

--- STAGE 3: ASPECT CLASSIFICATION ACCURACY ---
Aspect       | Oracle Accuracy      | E2E Accuracy        
-----------------------------------------------------------------
dim          | 85.38% (257/301)     | 59.63% (257/431)    
tr           | 70.43% (212/301)     | 49.19% (212/431)    
fr           | 61.79% (186/301)     | 43.16% (186/431)    
sr           | 79.40% (239/301)     | 55.45% (239/431)    
fe           | 92.03% (277/301)     | 64.27% (277/431)    
refLevel     | 99.67% (300/301)     | 69.61% (300/431)    
ss           | 98.34% (296/301)     | 68.68% (296/431)    
-----------------------------------------------------------------
Macro-Average  | 83.86%              | 58.57%
Strict Joint   | 37.54%              | 26.22%
